In [ ]:
from dotenv import load_dotenv

load_dotenv()

True

# Creating Sub-Agents

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the Square of a given Number"""
    return x ** 2

In [3]:
from langchain.agents import create_agent

#Create Sub-Agents

sub_agent_1 = create_agent(
    model="ollama:qwen2.5:3b",
    tools=[square_root]
)

sub_agent_2 = create_agent(
    model="ollama:qwen2.5:3b",
    tools=[square]
)

# Calling Sub-Agents

In [4]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = sub_agent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content
@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = sub_agent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

#Creating the main Agent

main_agent = create_agent(
    model="ollama:qwen2.5:3b",
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number."
)

# Testing Our Agents

In [5]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='fa1bd176-4d3b-4f08-9352-f1471f3046ed'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-14T18:10:09.18366Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5558851084, 'load_duration': 3534937125, 'prompt_eval_count': 222, 'prompt_eval_duration': 877475000, 'eval_count': 25, 'eval_duration': 1110311000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f61d2-5ace-7dc2-959d-e3e95c0d2c09-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': '13153fbc-7d98-4974-9880-d4cbb94ceb2d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 222, 'output_tokens': 25, 'total_tokens': 247}),
              ToolMessage(content='The square root of 456.0 is approximately 21.354.', name='call_subagent_1', id='26b5ea0b-3056-48e0

In [7]:
print(response["messages"][-1].content)

The square root of 456.0 is approximately 21.354.
